# 实验一 · LLM辅助评估清洗完成版

对应教师 `任务3_LLM辅助评估清洗.ipynb` 的工具、四模式和记忆任务。这里展示真实调用过的模型建议与评价，而非Mock；默认离线从raw复算已经发生的完整候选与记录选择，**不新增模型调用**。

系统包括治理工作流A/B/C与控制器、被测记录级四模式，两层成本分开。默认full恢复历史记忆搜索及模式合计6,001次记录—候选计算、384次原记录episode选择，并从本次新结果重新绘制模式图。最终统一/条件策略由G3真实结果冻结，不能把固定参数的全量处理继续叫逐条LLM。可用 `SC_LAB1_RECOMPUTE_WORK` 指定工程外的新输出目录；已有非空目录会被拒绝。

姓名：吴博闻；学号：`10245102410`。报告修订日期：2026-09-28；实验实际时间以各原始运行记录为准。当前为待审交付，用户理解审核尚未完成，未提交教师。

In [ ]:
from pathlib import Path
import contextlib, csv, gzip, hashlib, html, io, json, os, platform, re, sys, tempfile
from IPython.display import display, HTML, Image, Markdown

# 从 Notebook 所在位置或启动目录向上找项目；不依赖个人绝对路径和 ROOT/.venv。
start = Path.cwd().resolve()
ROOT = next((p for p in (start, *start.parents)
             if (p/'task1/goal3').is_dir()), None)
if ROOT is None:
    raise RuntimeError('请把 Notebook 放在完整提交包中，或从项目目录启动。')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
MODE = 'FULL_RECOMPUTE'
ENABLE_LIVE = False
assert MODE == 'FULL_RECOMPUTE' and ENABLE_LIVE is False
explicit_work = os.environ.get('SC_LAB1_RECOMPUTE_WORK')
if explicit_work:
    WORK = Path(explicit_work).expanduser().resolve()
    if WORK == ROOT or ROOT in WORK.parents:
        raise RuntimeError('显式复算目录必须位于当前工程/解压包之外。')
    if WORK.exists() and (not WORK.is_dir() or any(WORK.iterdir())):
        raise RuntimeError('显式复算目录已存在且非空，拒绝覆盖。')
    WORK.mkdir(parents=True, exist_ok=True)
else:
    WORK = Path(tempfile.mkdtemp(prefix='sc-lab1-full-')).resolve()

from task1.workflow.io import digest, object_hash, read_json, write_json
from task1.workflow.io import ROOT as MODULE_ROOT
from task1.workflow.g2_provider import ExperimentProvider
from task1.workflow.provider import CodexProvider
from task1.goal3.reproduce import recompute_historical, recompute_production
assert MODULE_ROOT.resolve() == ROOT, '模块必须来自当前工程/解压包，不能借用原工作区。'

# 本内核实际拦截两个已登记 Provider 入口；出现调用立即失败，不回退到 Mock。
provider_observation = {'attempted_new_calls': 0}
def forbid_provider(*args, **kwargs):
    provider_observation['attempted_new_calls'] += 1
    raise RuntimeError('FULL_RECOMPUTE 禁止新增模型调用')
_original_experiment_call = ExperimentProvider.call
_original_governance_call = CodexProvider.call
ExperimentProvider.call = forbid_provider
CodexProvider.call = forbid_provider

G2 = ROOT/'task1/evidence/goal2'
G3 = ROOT/'task1/evidence/goal3'
stage = read_json(G2/'result_summary.json')
memory_path = ROOT/stage['memory']['snapshot_path']
memory_hash_before = digest(memory_path)
assert memory_hash_before == stage['memory']['snapshot_sha256']
raw_path = ROOT/'task1/作业/作业/traj_dict.json'
assert digest(raw_path) == stage['data']['raw_sha256']

# 只有当前生产已冻结并有真实 run 时才执行本完成版，不临时用样本替代 full。
if not (G3/'current_runs.json').is_file() or not (G3/'production_freeze.json').is_file():
    raise RuntimeError('G3 正式生产尚未登记；保持源文件待执行，不把 fast 样本冒充 full。')

def show(rows, columns=None, limit=20):
    rows = list(rows)
    if not rows:
        print('无对应行；不填造数值。')
        return
    columns = columns or list(rows[0])
    def cell(value):
        if value is None: return '不可用（见相应原因）'
        if isinstance(value, (dict,list)): value=json.dumps(value,ensure_ascii=False)
        return html.escape(str(value))
    head='<tr>'+''.join('<th>'+cell(c)+'</th>' for c in columns)+'</tr>'
    body=''.join('<tr>'+''.join('<td>'+cell(r.get(c))+'</td>' for c in columns)+'</tr>' for r in rows[:limit])
    display(HTML('<table style="border-collapse:collapse" border="1">'+head+body+'</table>'))
    if len(rows)>limit: print(f'显示前 {limit} / {len(rows)} 行；完整记录保留于来源文件。')

def figure(name, phase='goal2', width=950):
    path=ROOT/'task1/figures'/phase/(name+'.png')
    if not path.is_file(): raise FileNotFoundError('正式图不存在：'+str(path.relative_to(ROOT)))
    display(Markdown('**已归档正式图：** '+phase+' / '+name+'（历史解释材料）'))
    display(Image(filename=str(path),width=width))

def recomputed_figure(topic, name, width=950):
    directory=WORK/topic/'figures'
    manifest=read_json(directory/'figure_manifest.json')
    assert manifest['old_summary_or_figure_read_for_plotting'] is False
    entry=next(f for f in manifest['figures'] if f['name']==name)
    path=directory/entry['files']['png']['path']
    assert digest(path)==entry['files']['png']['sha256']
    assert digest(directory/'plot_data.json')==manifest['plot_data_sha256']
    display(Markdown('**本次 FULL 从 raw 重新计算并绘制：** '+name))
    display(Image(filename=str(path),width=width))

def run_logged(label, fn):
    capture=io.StringIO()
    with contextlib.redirect_stdout(capture):
        result=fn()
    (WORK/(label+'_progress.txt')).write_text(capture.getvalue(),encoding='utf8')
    assert result['status']=='VERIFIED' and result['new_model_calls']==0
    print({k:v for k,v in result.items() if k not in {'checks','selections'}})
    return result

def current_manifest(partition):
    index=read_json(G3/'current_runs.json')
    found=[]
    for run_id in index.values():
        if not isinstance(run_id,str): continue
        path=G3/'runs'/run_id/'manifest.json'
        if path.is_file():
            value=read_json(path)
            if value.get('partition')==partition: found.append((path,value))
    if len(found)!=1: raise RuntimeError('当前 run 索引没有唯一分区：'+partition)
    return found[0]

_, deployment_manifest=current_manifest('FULL_PRODUCTION')
if (deployment_manifest.get('status')!='MACHINE_VERIFIED_PENDING_C'
        or deployment_manifest['input_ids']!=deployment_manifest['completed_record_ids']):
    raise RuntimeError('当前生产尚未完整完成；不执行待验收Notebook。')

print('FULL_RECOMPUTE；Python',platform.python_version(),'; 新模型调用由实际 Provider 哨兵检查。')
print('数据：',stage['data']['raw_records'],'条 /',stage['data']['raw_points'],'点；source_crs=UNVERIFIED。')

## 1. 工具与角色的真实边界

A根据可访问开发证据提出有限TaskPlan；B在固定代码/合同下运行；C从可信原始范围独立核验；确定性控制器维护依赖、预算、回执和恢复，不是第四个模型。普通实现缺陷走修复→独立复验→失效依赖重建→恢复父任务。

被测模式只调用允许的处理、诊断、评价、检索工具。Search only的记录级LLM调用为0，治理A不能实时暗示其参数。下图是历史G2实际结构，当前G3治理与最终生产接口见本轮Technical Handoff。

In [ ]:
figure('goal2_actual_architecture')
print('处理入口：task1.workflow.g2_pipeline.run_record')
print('可信审核：task1.workflow.g2_metrics.review_record')
print('有限选择：task1.workflow.g2_selection.select_record')
print('当前生产：task1.goal3.runtime.evaluate；默认离线复算 task1.goal3.reproduce')

## 2. 四模式与公平预算

| 模式 | 能看到的输入 | 真实结构 |
|---|---|---|
| LLM only | 原始诊断 | 一次提议，无搜索反馈/记忆，再实际评价 |
| Search only | 原始记录+冻结配置序列 | 最多20候选，0模型调用 |
| LLM + Search | 原始诊断+本记录允许反馈 | 最多3回合，累计8/14/20候选 |
| LLM + Memory + Search | 同上+冻结DEMO只读检索 | 相同预算，不写评估记录到记忆 |

每批最多8张诊断卡，一次batch请求只计一次调用。开发同24条×1episode，G2评估同24条×3episodes，不从三次中只挑最好一次。失败、非法原提议、回退全入分母，不能通过静默clamp把非法参数改成合法。

## 3. 从原始输入重新执行历史候选与选择

这一步逐条重算已保存真实提议对应的处理和评价，并核对原始锁定选择；仅复用真实模型建议本身，不请求模型重答。新模型调用由两个Provider入口的实际哨兵测量为0；记忆hash前后核对。

In [ ]:
mode_recompute=run_logged('modes',lambda:recompute_historical('modes',WORK/'modes'))
assert mode_recompute['candidate_record_recomputations']==6001
assert len(mode_recompute['selections'])==384
assert all(row['selection_match'] for row in mode_recompute['selections'])
recomputed_figure('modes','recomputed_modes')
show(stage['mode_summary'],['partition','mode','original_records','episodes','record_episode_observations',
                          'protected_gain_records_episodes','candidate_evaluations','experiment_model_dispatches',
                          'fallback_records_episodes','elapsed_batch_seconds_sum'])
assert all(r['experiment_model_dispatches']==0 for r in stage['mode_summary'] if r['mode']=='search-only')
figure('four_mode_paired_results')
figure('four_mode_failures_and_cost')

## 4. 原样建议、核验和拒绝

模型原话被保存，并在执行前检查参数域、预算与权限。合法建议仍可能因共同覆盖或几何保护失败而不被采用。有效开发148个原提议中70个方向预测可评价且一致，其余有不可评价原因；没有观察到非法提议/方向预测错误/回退，不能为展示批判制造事件。真实存在的4个合法权衡建议完整保留。

下面展示真实response来源和具体比较，尤其留意记录7764的同集合误差退化、失去11个原覆盖身份后null的含义，以及记录10232正确预测删除数下降的正例。

In [ ]:
ai=read_json(G2/'a_epoch02/actual_ai_critique/actual_ai_critique.json')
show([{'id':x['citation_id'],'record':x['record_id'],'reason_verbatim':x['verbatim_model_reason'],
       'parameters':x['candidate_parameters'],'legal':x['legal'],'executed':x['executed'],
       'selected':x['proposal_became_locked_output'],'raw_response':x['raw_response']}
      for x in ai['actual_ai_citations']],limit=6)
show([{'id':x['citation_id'],
       'reference_coverage':x['reference_metrics']['common_covered_points'],
       'candidate_coverage':x['candidate_metrics']['common_covered_points'],
       'reference_common_max':x['reference_metrics']['common_max_error'],
       'candidate_own_common_max':x['candidate_metrics']['common_max_error'],
       'same_common_covered_mask':(x['registered_comparison'] or {}).get('same_common_covered_point_mask'),
       'protection_readings':(x['registered_comparison'] or {}).get('registered_protection_readings')}
      for x in ai['actual_ai_citations'][:3]])
print('覆盖集合不同的两个自身最大值不作配对误差增减；真实数据无噪声真值。')

## 5. 只读示范记忆：送达、引用、动作与因果分开

DEMO60条用于记忆，其中26条范围内支持、34条无新增收益。六个原始诊断特征只用DEMO缩放；描述层内最多3近邻，排除自身/关联组。评估冻结只读，工作记忆按记录episode隔离，用户知识条目实际为0。

合计96次带记忆记录观察均送达检索；241个决策回合中60次引用、49次动作一致。分母不同。评估带/不带记忆搜索各47/72支持输出，不能因此宣称因果收益或普遍等效。

In [ ]:
print('只读记忆hash：',memory_hash_before)
show([stage['memory']],['demonstration_records','research_status_counts','read_only_policy','human_knowledge_entries','work_memory_persisted'])
with (G2/'tables/memory_consumption.csv').open(encoding='utf8',newline='') as stream:
    consumption=list(csv.DictReader(stream))
print('完整记忆消费记录行数：',len(consumption))
show(consumption,limit=8)
figure('memory_coverage_and_consumption')
assert digest(memory_path)==memory_hash_before

## 6. 真正的新调用与成本

历史有效实验模型调用84次；历史失效/中断15次、工程资格1次另记。模型为既有Codex通道gpt-6-astra/medium，没有固定seed或精确snapshot。隐藏底层请求和货币费用不可见时保持unknown，不填0。

本Notebook只复算。需要新LIVE时必须创建新run、使用已授权provider和完整预算/权限/回执入口，不能把此处字符串改成LIVE就静默发请求。下面只用当前Python解释器查看CLI帮助，不调用provider。

In [ ]:
import subprocess
help_result=subprocess.run([sys.executable,'-m','task1.scripts.goal2','--help'],cwd=ROOT,
                           text=True,capture_output=True,check=True)
print(help_result.stdout)
resources=stage['resource_accounting']
show([{'scope':'current_valid',**resources['current_valid_model_costs']},
      {'scope':'historical_superseded_or_interrupted',**resources['historical_model_costs']}],
     ['scope','visible_experiment_model_dispatches','completed_calls','interrupted_or_failed_calls',
      'provider_request_count','currency_cost','counts_as_current_effectiveness_evidence'])

## 7. 当前冻结生产策略与系统选择

当前结果从生产freeze与真实run读取。统一确定性配置、简单规则或逐条搜索/模型是记录级算法身份；治理使用A/B/C不改变该身份。最终确认和生产记忆保持只读，不使用最终评分更新参数或规则。全量处理与最终确认原始记录分母不同。

In [ ]:
production_path,production=current_manifest('FULL_PRODUCTION')
show([{'strategy':cid,'parameters':value['parameters'],'order':value['order'],
       'conditional_rule':value.get('conditional_rule')} for cid,value in production['strategies'].items()])
print('全量生产原始记录:',len(production['input_ids']))
confirm_path,confirmation=current_manifest('FINAL_CONFIRM')
print('一次性最终确认原始记录:',len(confirmation['input_ids']))
show([{'pair':key,**value} for key,value in confirmation['comparisons'].items()])
figure('goal3_actual_workflow',phase='goal3')
print('实际独立核验、恢复和版本入口：task1/evidence/goal3/REVIEW_PACKET.md')

## 8. 交付与真实互动边界

技术报告给方法与结果，Process Report保留Workflow Construction和Experiment Decision Process两层。缺真实截图、Evidence Master规格或Lock时文档标待补，不能用Agent互评代替人类判断。用户Understanding和网页GPT二重审核不由Notebook自动设PASS。教师包实际准备但不代发邮件。

In [ ]:
receipt={'notebook':'llm_system','historical_record_candidate_recomputations':mode_recompute['candidate_record_recomputations'],
         'recomputed_original_record_episode_selections':len(mode_recompute['selections']),
         'new_mode_plots':mode_recompute['raw_recomputed_plots']}

assert digest(memory_path)==memory_hash_before
assert digest(raw_path)==stage['data']['raw_sha256']
assert provider_observation['attempted_new_calls']==0
ExperimentProvider.call=_original_experiment_call
CodexProvider.call=_original_governance_call
receipt.update({'status':'VERIFIED','mode':MODE,'python_version':platform.python_version(),
                'new_model_call_attempts_observed':provider_observation['attempted_new_calls'],
                'memory_sha256_before':memory_hash_before,'memory_sha256_after':digest(memory_path),
                'raw_sha256_unchanged':digest(raw_path)==stage['data']['raw_sha256'],
                'historical_recompute_is_new_live':False,
                'execution_artifact_directory_name':WORK.name,
                'execution_directory_source':'SC_LAB1_RECOMPUTE_WORK' if explicit_work else 'new_temporary_directory',
                'module_root_matches_current_project':MODULE_ROOT.resolve()==ROOT})
write_json(WORK/'notebook_receipt.json',receipt)
print(json.dumps(receipt,ensure_ascii=False,indent=2))
print('本次新模型调用0；历史提议复算不作为新LIVE。网页GPT二重审核仍由外部实际审核决定。')